In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQ_F.xpt"
csv_file = DATA_DIR / "SMQ_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 SMQ_F XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SMQ020",
    "SMD030",
    "SMQ040",
    "SMQ050Q",
    "SMQ050U",
    "SMD055",
    "SMD057",
    "SMQ077",
    "SMD641",
    "SMD650",
    "SMD093",
    "SMDUPCA",
    "SMD100BR",
    "SMD100FL",
    "SMD100MN",
    "SMD100LN",
    "SMD100TR",
    "SMD100NI",
    "SMD100CO",
    "SMQ620",
    "SMD630",
    "SMQ660",
    "SMQ664M",
    "SMQ664C",
    "SMQ664W",
    "SMQ664B",
    "SMQ664O",
    "SMQ666M",
    "SMQ666C",
    "SMQ666K",
    "SMQ666W",
    "SMQ666B",
    "SMQ666S",
    "SMQ666O",
    "SMQ670",
    "SMAQUEX2"
]


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "SMQ_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 7528:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 37:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 7,528 rows x 37 columns."
)


# ============================================================
# 4. DEFINE VARIABLE TYPES
# ============================================================

#
# SMQ_F is NOT an all-integer questionnaire file.
#
# Two fields are text:
#
#   SMDUPCA  = 12-character cigarette UPC
#   SMD100BR = cigarette brand/sub-brand
#
# One field is a genuine decimal measurement:
#
#   SMD100NI = FTC nicotine content
#
# All other numeric fields are whole-number
# questionnaire codes/counts.
#

STRING_COLUMNS = [
    "SMDUPCA",
    "SMD100BR"
]


DECIMAL_COLUMNS = [
    "SMD100NI"
]


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


INTEGER_COLUMNS = [
    col
    for col in numeric_cols
    if col not in DECIMAL_COLUMNS
]


print("\nString columns:")
print(STRING_COLUMNS)

print("\nDecimal columns:")
print(DECIMAL_COLUMNS)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Attached SMQ_F.xpt:
#
# SMD030   = 97
# SMD641   = 132
# SMD100FL = 14
# SMD100MN = 867
#
# Total = 1,110
#
#
# CDC validation:
#
# SMD030
#   0 = Never smoked cigarettes regularly
#
# SMD641
#   valid range = 0-30 days
#
# SMD100FL
#   0 = Non-filter
#
# SMD100MN
#   0 = Non-menthol
#

EXPECTED_TINY_COUNTS = {
    "SMD030": 97,
    "SMD641": 132,
    "SMD100FL": 14,
    "SMD100MN": 867
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:
    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated SMQ_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 1110:
    raise ValueError(
        f"Unexpected total tiny count: {tiny_total}"
    )


print(
    "\nPASS: Exact 1,110-value tiny-zero pattern validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    tiny_mask = df[col].eq(TINY_VALUE)

    actual_count = int(
        tiny_mask.sum()
    )

    if actual_count != expected_count:
        raise ValueError(
            f"{col}: unexpected tiny-value count."
        )


    df.loc[
        tiny_mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 1110:
    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# CRITICAL:
#
# This check is done AFTER validated tiny-zero restoration
# but BEFORE Int64 conversion.
#
# Expected genuine decimal column:
#
#     SMD100NI
#
# It must remain float and must NOT be rounded.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "fractional observations"
        )


if fractional_columns != ["SMD100NI"]:

    raise ValueError(
        "Unexpected decimal-column pattern.\n"
        "Expected: ['SMD100NI']\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "PASS: SMD100NI is the only genuine "
    "decimal variable."
)


# ============================================================
# 10. VALIDATE SMD100NI DECIMAL VALUES
# ============================================================

#
# CDC:
#
# FTC nicotine content
# Valid range = 0.1 to 1.8
#
# Attached XPT:
#
# nonmissing = 922
# missing    = 6,606
#
# IMPORTANT:
#
# Do NOT round this variable.
#

nicotine = df["SMD100NI"]


print("\n--- SMD100NI ---")

print(
    "Minimum:",
    nicotine.min()
)

print(
    "Maximum:",
    nicotine.max()
)

print(
    "Non-missing:",
    int(
        nicotine.notna()
        .sum()
    )
)

print(
    "Missing:",
    int(
        nicotine.isna()
        .sum()
    )
)


if not (
    nicotine.dropna()
    .between(
        0.1,
        1.8
    )
    .all()
):
    raise ValueError(
        "SMD100NI contains values outside "
        "CDC range 0.1-1.8."
    )


if int(
    nicotine.notna()
    .sum()
) != 922:
    raise ValueError(
        "Unexpected SMD100NI non-missing count "
        "in attached XPT."
    )


print(
    "PASS: SMD100NI preserved as genuine decimal."
)


# ============================================================
# 11. VALIDATE TEXT FIELDS
# ============================================================

#
# IMPORTANT:
#
# SMDUPCA MUST remain text.
#
# Example:
#
#     027200008544
#
# If converted to numeric:
#
#     27200008544
#
# the leading zero would be lost.
#

print("\n--- TEXT FIELD CHECK ---")


for col in STRING_COLUMNS:

    if not df[col].map(
        lambda x: isinstance(x, str)
    ).all():

        raise ValueError(
            f"{col} contains unexpected non-string values."
        )


# ---------- SMDUPCA ----------

upc_nonempty = (
    df["SMDUPCA"]
    .ne("")
)


upc_count = int(
    upc_nonempty.sum()
)


print(
    "SMDUPCA populated:",
    upc_count
)


if upc_count != 757:
    raise ValueError(
        "Unexpected SMDUPCA populated count."
    )


upc_lengths = (
    df.loc[
        upc_nonempty,
        "SMDUPCA"
    ]
    .str.len()
)


if not upc_lengths.eq(12).all():
    raise ValueError(
        "SMDUPCA contains a populated value "
        "that is not exactly 12 characters."
    )


print(
    "PASS: All populated SMDUPCA values "
    "are exactly 12 characters."
)


# ---------- SMD100BR ----------

brand_nonempty = (
    df["SMD100BR"]
    .ne("")
)


brand_count = int(
    brand_nonempty.sum()
)


print(
    "SMD100BR populated:",
    brand_count
)


if brand_count != 1272:
    raise ValueError(
        "Unexpected SMD100BR populated count."
    )


print(
    "PASS: String fields validated."
)


# ============================================================
# 12. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):
    raise ValueError(
        "SEQN contains unexpected fractions."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 51624
    and
    df["SEQN"].max() == 62160
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):
    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 13. FORMAT ONLY PROVEN WHOLE-NUMBER NUMERIC FIELDS
# ============================================================

#
# DO NOT convert:
#
#     SMD100NI
#
# DO NOT touch:
#
#     SMDUPCA
#     SMD100BR
#
# Every other numeric field is proven whole-number.
#
# NO .round() IS USED.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():
            raise ValueError(
                f"{col} contains genuine decimals "
                "and cannot be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Whole-number fields converted to Int64."
)

print(
    "PASS: SMD100NI remains decimal."
)

print(
    "PASS: SMDUPCA/SMD100BR remain text."
)


# ============================================================
# 14. HELPER FOR CDC FREQUENCY CHECKS
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:
        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:
        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 15. CDC CHECK - SMQ020
# ============================================================

check_frequency(
    "SMQ020",
    {
        1: 2866,
        2: 3352
    },
    1310
)


# ============================================================
# 16. CDC CHECK - SMD030
# ============================================================

#
# Age started smoking regularly.
#
# CDC:
#
# 6-72 = 2,757
# 0    = Never regularly smoked = 97
# 777  = 1
# 999  = 11
# Missing = 4,662
#

s = df["SMD030"]


if not (
    int(
        s.between(
            6,
            72
        ).sum()
    ) == 2757
    and
    int(
        s.eq(0)
        .sum()
    ) == 97
    and
    int(
        s.eq(777)
        .sum()
    ) == 1
    and
    int(
        s.eq(999)
        .sum()
    ) == 11
    and
    int(
        s.isna()
        .sum()
    ) == 4662
):
    raise ValueError(
        "SMD030 differs from CDC / zero restoration."
    )


print(
    "PASS: SMD030 matches CDC."
)


# ============================================================
# 17. CDC CHECK - SMQ040
# ============================================================

check_frequency(
    "SMQ040",
    {
        1: 1103,
        2: 243,
        3: 1520
    },
    4662
)


# ============================================================
# 18. CDC CHECK - SMQ050Q / SMQ050U
# ============================================================

s = df["SMQ050Q"]


if not (
    int(
        s.between(
            1,
            67
        ).sum()
    ) == 1512
    and
    int(
        s.eq(77777)
        .sum()
    ) == 0
    and
    int(
        s.eq(99999)
        .sum()
    ) == 8
    and
    int(
        s.isna()
        .sum()
    ) == 6008
):
    raise ValueError(
        "SMQ050Q differs from CDC."
    )


print(
    "PASS: SMQ050Q matches CDC."
)


check_frequency(
    "SMQ050U",
    {
        1: 12,
        2: 21,
        3: 103,
        4: 1376
    },
    6016
)


# ============================================================
# 19. CDC CHECK - SMD641
# ============================================================

#
# Number of days smoked in last 30 days.
#
# CDC valid range = 0-30.
#
# Attached XPT:
#
# zero = 132
#
# CDC publishes 1,636 total observations
# in the 0-30 range.
#

s = df["SMD641"]


print("\n--- SMD641 ---")

print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            0,
            30
        ).sum()
    ) == 1636
    and
    int(
        s.eq(0)
        .sum()
    ) == 132
    and
    int(
        s.eq(77)
        .sum()
    ) == 0
    and
    int(
        s.eq(99)
        .sum()
    ) == 5
    and
    int(
        s.isna()
        .sum()
    ) == 5887
):
    raise ValueError(
        "SMD641 differs from CDC / zero restoration."
    )


print(
    "PASS: SMD641 matches CDC range/frequencies."
)


# ============================================================
# 20. CDC CHECK - CIGARETTE PACK / PRODUCT FIELDS
# ============================================================

check_frequency(
    "SMD093",
    {
        1: 757,
        2: 536,
        3: 27,
        4: 26
    },
    6182
)


check_frequency(
    "SMD100FL",
    {
        0: 14,
        1: 1257
    },
    6257
)


check_frequency(
    "SMD100MN",
    {
        0: 867,
        1: 404
    },
    6257
)


check_frequency(
    "SMD100LN",
    {
        1: 223,
        2: 612,
        3: 423,
        4: 13
    },
    6257
)


# ============================================================
# 21. CDC RANGE CHECKS - TAR / NICOTINE / CO
# ============================================================

#
# IMPORTANT:
#
# Current CDC webpage lists 1,190 nonmissing values
# for SMD100TR, SMD100NI and SMD100CO.
#
# The attached XPT contains 922 nonmissing values
# for each.
#
# We DO NOT alter the attached XPT to manufacture
# the additional values.
#
# We validate that existing released values remain
# inside the CDC published ranges and preserve them
# exactly.
#

PRODUCT_RANGES = {
    "SMD100TR": (1, 24),
    "SMD100NI": (0.1, 1.8),
    "SMD100CO": (1, 22)
}


print("\n--- PRODUCT MEASUREMENT RANGE CHECK ---")


for col, (minimum, maximum) in PRODUCT_RANGES.items():

    values = (
        df[col]
        .dropna()
    )


    print(
        col,
        "non-missing:",
        f"{len(values):,}",
        "range:",
        values.min(),
        "to",
        values.max()
    )


    if not values.between(
        minimum,
        maximum
    ).all():
        raise ValueError(
            f"{col} has values outside CDC range."
        )


print(
    "PASS: Existing tar/nicotine/CO values "
    "are within CDC ranges."
)


# ============================================================
# 22. CDC CHECK - YOUTH VARIABLES
# ============================================================

check_frequency(
    "SMQ620",
    {
        1: 402,
        2: 793,
        7: 1
    },
    6332
)


check_frequency(
    "SMQ660",
    {
        1: 69,
        2: 24,
        3: 57,
        4: 3,
        8: 17
    },
    7358
)


check_frequency(
    "SMQ664M",
    {
        1: 29,
        2: 40
    },
    7459
)


check_frequency(
    "SMQ664C",
    {
        1: 14,
        2: 10
    },
    7504
)


check_frequency(
    "SMQ664O",
    {
        1: 5,
        2: 12
    },
    7511
)


check_frequency(
    "SMQ666M",
    {
        1: 48,
        2: 18,
        3: 3
    },
    7459
)


check_frequency(
    "SMQ666C",
    {
        1: 16,
        2: 8
    },
    7504
)


check_frequency(
    "SMQ666K",
    {
        1: 3
    },
    7525
)


check_frequency(
    "SMQ666O",
    {
        1: 13,
        2: 3,
        3: 1
    },
    7511
)


check_frequency(
    "SMQ670",
    {
        1: 159,
        2: 134,
        9: 2
    },
    7233
)


# ============================================================
# 23. CDC CHECK - QUESTIONNAIRE MODE
# ============================================================

#
# CDC:
#
# 1 = Home Interview (20+) = 6,218
# 2 = A-CASI (12-19)       = 1,310
#

check_frequency(
    "SMAQUEX2",
    {
        1: 6218,
        2: 1310
    },
    0
)


# ============================================================
# 24. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected numeric changes ONLY:
#
# SMD030:
#     97 tiny -> 0
#
# SMD641:
#     132 tiny -> 0
#
# SMD100FL:
#     14 tiny -> 0
#
# SMD100MN:
#     867 tiny -> 0
#
# TOTAL = 1,110
#
# SMD100NI must remain exactly unchanged.
#
# SMDUPCA and SMD100BR must remain exact strings.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


numeric_failures = []


for col in numeric_cols:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):
        numeric_failures.append(
            col
        )


if numeric_failures:
    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(numeric_failures)
    )


string_failures = []


for col in STRING_COLUMNS:

    expected_strings = (
        expected_df[col]
        .fillna("")
        .astype(str)
        .to_numpy()
    )

    current_strings = (
        df[col]
        .fillna("")
        .astype(str)
        .to_numpy()
    )


    if not np.array_equal(
        expected_strings,
        current_strings
    ):
        string_failures.append(
            col
        )


if string_failures:
    raise ValueError(
        "Unexpected text changes before export:\n"
        + str(string_failures)
    )


print(
    "PASS: Numeric and text values match "
    "expected corrected XPT exactly."
)


# ============================================================
# 25. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO float_format
# NO rounding
#
# This preserves SMD100NI as its original
# decimal numeric value.
#
# SMDUPCA remains a string, preserving leading zeros.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 26. RE-READ CSV AS RAW TEXT
# ============================================================

#
# Read all fields as strings first so that:
#
#     027200008544
#
# stays exactly:
#
#     027200008544
#
# instead of becoming:
#
#     27200008544
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != df.shape:
    raise ValueError(
        "CSV dimensions changed after export."
    )


if raw_csv_df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 27. EXACT NUMERIC XPT -> CSV FIDELITY
# ============================================================

print("\n--- NUMERIC XPT -> CSV FIDELITY ---")


csv_numeric_failures = []


for col in numeric_cols:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    raw_values = (
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        )
    )


    exported_values = (
        pd.to_numeric(
            raw_values,
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):
        csv_numeric_failures.append(
            col
        )


if csv_numeric_failures:
    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_numeric_failures)
    )


print(
    "PASS: All numeric values round-trip exactly."
)


# ============================================================
# 28. EXACT STRING FIDELITY
# ============================================================

print("\n--- STRING FIDELITY ---")


for col in STRING_COLUMNS:

    expected_strings = (
        expected_df[col]
        .fillna("")
        .astype(str)
        .to_numpy()
    )


    exported_strings = (
        raw_csv_df[col]
        .to_numpy()
    )


    if not np.array_equal(
        expected_strings,
        exported_strings
    ):
        raise ValueError(
            f"{col}: string values changed "
            "during CSV export."
        )


    print(
        f"PASS: {col} preserved exactly."
    )


# ============================================================
# 29. UPC LEADING-ZERO / WIDTH CHECK
# ============================================================

csv_upc_nonempty = (
    raw_csv_df["SMDUPCA"]
    .ne("")
)


if int(
    csv_upc_nonempty.sum()
) != 757:
    raise ValueError(
        "SMDUPCA populated count changed."
    )


if not (
    raw_csv_df.loc[
        csv_upc_nonempty,
        "SMDUPCA"
    ]
    .str.len()
    .eq(12)
    .all()
):
    raise ValueError(
        "SMDUPCA width changed."
    )


print(
    "PASS: SMDUPCA remains 12-character text."
)


print(
    "Example UPC values:"
)

print(
    raw_csv_df.loc[
        csv_upc_nonempty,
        "SMDUPCA"
    ]
    .head()
    .tolist()
)


# ============================================================
# 30. MISSING VALUE FIDELITY
# ============================================================

#
# Numeric blanks in CSV must correspond exactly
# to XPT numeric NaN positions.
#

missing_mismatches = 0


for col in numeric_cols:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        raw_csv_df[col]
        .eq("")
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Numeric missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:
    raise ValueError(
        "Numeric missing-value positions changed."
    )


print(
    "PASS: Numeric missing positions preserved exactly."
)


# ============================================================
# 31. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in numeric_cols:

    values = (
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        )
    )


    numeric_values = pd.to_numeric(
        values,
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:
    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 32. .0 FORMAT CHECK - INTEGER COLUMNS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# Do NOT search the entire CSV for ".0".
#
# SMD100NI is a genuine decimal field.
#
# For example:
#
#     1.0
#
# is a legitimate nicotine measurement.
#
# Only proven integer columns should be checked
# for unnecessary ".0".
#

print("\n--- INTEGER COLUMN .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]

    bad_mask = (
        tokens
        .ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:
        dot_zero_errors[col] = count


if dot_zero_errors:
    raise ValueError(
        "Unexpected .0 formatting in integer fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 in integer columns."
)


#
# This is allowed because SMD100NI is a decimal field.
#

nicotine_dot_zero = int(
    raw_csv_df["SMD100NI"]
    .str.endswith(".0")
    .sum()
)


print(
    "Legitimate SMD100NI values displayed with .0:",
    nicotine_dot_zero
)


# ============================================================
# 33. EXACT DECIMAL FIDELITY
# ============================================================

expected_nicotine = (
    expected_df["SMD100NI"]
    .to_numpy(
        dtype=float
    )
)


exported_nicotine = (
    pd.to_numeric(
        raw_csv_df["SMD100NI"]
        .where(
            raw_csv_df["SMD100NI"] != "",
            np.nan
        ),
        errors="coerce"
    )
    .to_numpy(
        dtype=float
    )
)


if not np.allclose(
    expected_nicotine,
    exported_nicotine,
    rtol=0,
    atol=0,
    equal_nan=True
):
    raise ValueError(
        "SMD100NI decimal fidelity failed."
    )


print(
    "PASS: SMD100NI decimals preserved exactly."
)


# ============================================================
# 34. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:
    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 35. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 51624
    and
    final_seqn.max() == 62160
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):
    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN valid."
)


# ============================================================
# 36. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)

print(
    "SMQ_F CONVERSION COMPLETE"
)

print(
    "=" * 75
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Original tiny artifacts:",
    f"{tiny_total:,}"
)

print(
    "SMD030 zero restorations:",
    97
)

print(
    "SMD641 zero restorations:",
    132
)

print(
    "SMD100FL zero restorations:",
    14
)

print(
    "SMD100MN zero restorations:",
    867
)

print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "String variables:",
    STRING_COLUMNS
)

print(
    "SMDUPCA leading zeros preserved: YES"
)

print(
    "Unexpected numeric differences: 0 expected"
)

print(
    "String differences: 0 expected"
)

print(
    "Missing-position differences: 0 expected"
)

print(
    "Tiny values remaining: 0 expected"
)

print(
    "Unnecessary .0 in integer fields: 0 expected"
)

print(
    "Scientific notation: 0 expected"
)

NHANES 2009-2010 SMQ_F XPT -> CSV
Rows: 7,528
Columns: 37
PASS: Structure = 7,528 rows x 37 columns.

String columns:
['SMDUPCA', 'SMD100BR']

Decimal columns:
['SMD100NI']

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 1,110

Tiny values by variable:
SMD100MN    867
SMD641      132
SMD030       97
SMD100FL     14

PASS: Exact 1,110-value tiny-zero pattern validated.

--- ZERO RESTORATION ---
SMD030: 97 restored to 0
SMD641: 132 restored to 0
SMD100FL: 14 restored to 0
SMD100MN: 867 restored to 0

PASS: 1,110 validated zero values restored.

--- FRACTIONAL VALUE CHECK ---
SMD100NI: 845 fractional observations
PASS: SMD100NI is the only genuine decimal variable.

--- SMD100NI ---
Minimum: 0.1
Maximum: 1.8
Non-missing: 922
Missing: 6606
PASS: SMD100NI preserved as genuine decimal.

--- TEXT FIELD CHECK ---
SMDUPCA populated: 757
PASS: All populated SMDUPCA values are exactly 12 characters.
SMD100BR populated: 1272
PASS: String fields validated.

--- SEQ